# articleだけでLGBMで予測してCBと比較

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier

# ---------------------
# ① データ読み込み
# ---------------------
transactions = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/transactions_train.csv")
customers = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/customers.csv")
articles = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/articles.csv")
sample_submission = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/sample_submission.csv")

In [2]:
# ---------------------
# ② 正例（直近購入）・負例（ランダム）
# ---------------------
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
positive = transactions[transactions['t_dat'] >= '2020-09-15'][['customer_id', 'article_id']].drop_duplicates()
positive['target'] = 1

# 負例作成（1:5の比率）
np.random.seed(42)
neg = pd.DataFrame({
    'customer_id': np.random.choice(positive['customer_id'].unique(), size=len(positive)*5),
    'article_id': np.random.choice(positive['article_id'].unique(), size=len(positive)*5),
})
neg['target'] = 0

# 正例と重複排除
neg = neg[~neg.set_index(['customer_id', 'article_id']).index.isin(
    positive.set_index(['customer_id', 'article_id']).index)]

# 結合 & マージ
data = pd.concat([positive, neg], ignore_index=True)
data = data.merge(articles, on='article_id', how='left')
data = data.merge(customers, on='customer_id', how='left')


In [3]:
# ---------------------
# ③ 特徴量とターゲット分離
# ---------------------
drop_cols = ['customer_id', 'article_id', 'target']
X = data.drop(columns=drop_cols)
y = data['target']

# カテゴリ変数をcategory型に
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].astype('category')

# ---------------------
# ④ モデル学習
# ---------------------
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = LGBMClassifier(random_state=42)
model.fit(X_train, y_train)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 189298, number of negative: 946314
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009346 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 54924
[LightGBM] [Info] Number of data points in the train set: 1135612, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166692 -> initscore=-1.609252
[LightGBM] [Info] Start training from score -1.609252


LGBMClassifier(random_state=42)

In [4]:
# ---------------------
# ⑤ スコア予測
# ---------------------
data['pred_score'] = model.predict_proba(X)[:, 1]

# ---------------------
# ⑥ 上位12件推薦（customer_id単位）
# ---------------------
top12 = (
    data.groupby('customer_id')
    .apply(lambda x: x.sort_values('pred_score', ascending=False).head(12))
    .reset_index(drop=True)
)

/var/folders/bf/5s1y_hg57vldk9jqhsj92l8h0000gn/T/ipykernel_1043/2296534393.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values('pred_score', ascending=False).head(12))


In [5]:
# customer_idごとに推薦されたarticle_idの数をカウント
recommendation_counts = (
    top12.groupby('customer_id')['article_id']
    .count()
    .reset_index(name='num_recommendations')
)

# 12個未満の推薦しかされていないcustomer_idを抽出
under_12 = recommendation_counts[recommendation_counts['num_recommendations'] < 12]

# 件数と一部のデータを表示
print(f"✅ 12個未満の推薦がある顧客数: {len(under_12)}")
display(under_12.head())


✅ 12個未満の推薦がある顧客数: 3475


,customer_id,num_recommendations
20,001c1f8d70782f450524d3b3f404474dbd4a7d0d2ad78a...,10
26,0024dbfbe9bf4f2db0ec54e1ce39ecfa91759fd29fa556...,11
49,0038518fc66b22eafe13b0e80c5bf8e13e85c03e434a2c...,10
90,0058caa8e25099d45288038fe4b23efbe030c41277f76a...,11
102,0063a5fab642a52b80dcc5561b3a2ef5a06f13f2967a7a...,11


In [6]:
# ---------------------
# ⑦ 不足分を補完（人気商品）
# ---------------------
# 人気商品（10桁ゼロ埋め）
popular_articles = transactions['article_id'].value_counts().head(30).index.astype(str).str.zfill(10).tolist()

def recommend_12_articles(article_list):
    article_list = [str(a).zfill(10) for a in article_list]
    if len(article_list) >= 12:
        return article_list[:12]
    else:
        fill = [a for a in popular_articles if a not in article_list]
        return article_list + fill[:12 - len(article_list)]

submission_pred = (
    top12.groupby('customer_id')['article_id']
    .apply(lambda x: ' '.join(recommend_12_articles(x)))
    .reset_index()
)


In [7]:
# ---------------------
# ⑧ 提出ファイル作成
# ---------------------
submission = sample_submission[['customer_id']].merge(submission_pred, on='customer_id', how='left')
fallback_12 = ' '.join(popular_articles[:12])
submission['prediction'] = submission['article_id'].fillna(fallback_12)
submission.drop(columns='article_id', inplace=True)
submission.to_csv("submission_lgbm.csv", index=False)
print("✅ submission_lgbm.csv を保存しました")

✅ submission_lgbm.csv を保存しました


In [9]:
print(submission.shape)
submission.head()

(1371980, 2)


,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0706016001 0706016002 0372860001 0610776002 07...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0706016001 0706016002 0372860001 0610776002 07...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0804992034 0827957002 0794321007 0881112001 08...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0706016001 0706016002 0372860001 0610776002 07...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0706016001 0706016002 0372860001 0610776002 07...
